# 31 — Score overlap and heterogeneous lower-back rescue

This notebook tests whether the remaining FP and FN can be reduced without threshold manipulation or external-data leakage. RevalExo and NONAN are not loaded.

## Predeclared sequence

1. Quantify the complete participant-level threshold frontier for the selected lower-back deep ensemble.
2. Fit a lower-back MiniROCKET transform plus weighted logistic classifier under the same five seeds and three complete source holdouts. MiniROCKET bias windows are balanced by source, class, and participant.
3. Form one fixed 50:50 heterogeneous probability ensemble from the accepted deep ensemble and MiniROCKET.
4. Admit it only if worst-source discrimination/calibration remain non-inferior, both mean FP and mean FN do not increase, and mean total errors fall by at least 10%.

The motivation is representation diversity rather than architecture novelty. [MiniROCKET](https://github.com/angus924/minirocket) provides almost-deterministic random convolutional features. [HIVE-COTE 2.0](https://link.springer.com/article/10.1007/s10994-021-06057-9) supports combining classifiers built from different time-series representations, while warning that combination quality must be estimated using training-side evidence rather than the final test.


In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.models.lower_back_minirocket_rescue import run

module_text = (ROOT/'src/models/lower_back_minirocket_rescue.py').read_text(encoding='utf-8').lower()
assert 'revalexo_external_windows' not in module_text and 'nonan_external_windows' not in module_text
print('Frozen external signal loaders referenced: NONE')


Frozen external signal loaders referenced: NONE


In [2]:
pred = pd.read_csv(ROOT/'data/processed/evidence_gated_lower_back_dg_participant_predictions.csv')
people = pred.query("method == 'ensemble_all'").groupby(['source','group','y'], as_index=False).probability.mean()
thresholds = np.r_[-1e-9, np.sort(people.probability.unique()), 1+1e-9]
rows=[]
for threshold in thresholds:
    label = people.probability.ge(threshold).astype(int)
    fp=int(((people.y==0)&(label==1)).sum()); fn=int(((people.y==1)&(label==0)).sum())
    rows.append({'threshold':threshold,'FP':fp,'FN':fn,'total_errors':fp+fn,
                 'specificity':1-fp/(people.y==0).sum(),'sensitivity':1-fn/(people.y==1).sum()})
frontier=pd.DataFrame(rows)
display(frontier.sort_values(['total_errors','threshold']).head(10).round(4))
display(frontier.query('FP == 0').sort_values('FN').head(1).round(4))
display(frontier.query('FN == 0').sort_values('FP').head(1).round(4))
print('Maximum healthy score:', people.loc[people.y.eq(0),'probability'].max())
print('Minimum stroke score:', people.loc[people.y.eq(1),'probability'].min())


,threshold,FP,FN,total_errors,specificity,sensitivity
95,0.3179,45,13,58,0.6429,0.9309
129,0.4656,28,30,58,0.7778,0.8404
131,0.4782,27,31,58,0.7857,0.8351
94,0.3118,46,13,59,0.6349,0.9309
96,0.3242,45,14,59,0.6429,0.9255
126,0.4584,30,29,59,0.7619,0.8457
128,0.4616,29,30,59,0.7698,0.8404
130,0.4668,28,31,59,0.7778,0.8351
132,0.4849,27,32,59,0.7857,0.8298
89,0.2753,49,11,60,0.6111,0.9415


,threshold,FP,FN,total_errors,specificity,sensitivity
284,0.9719,0,157,157,1.0,0.1649


,threshold,FP,FN,total_errors,specificity,sensitivity
40,0.101,87,0,87,0.3095,1.0


Maximum healthy score: 0.971781346
Minimum stroke score: 0.10099002320000001


In [3]:
rescue = run(ROOT)


MiniROCKET complete: felius_2024 seed=42 BA=0.730 FP=6 FN=47


MiniROCKET complete: felius_2024 seed=137 BA=0.726 FP=6 FN=48


MiniROCKET complete: felius_2024 seed=202 BA=0.722 FP=6 FN=49


MiniROCKET complete: felius_2024 seed=314 BA=0.726 FP=6 FN=48


MiniROCKET complete: felius_2024 seed=515 BA=0.726 FP=6 FN=48


...

MiniROCKET complete: voisard_2025 seed=42 BA=0.742 FP=21 FN=11


MiniROCKET complete: voisard_2025 seed=137 BA=0.766 FP=16 FN=12


MiniROCKET complete: voisard_2025 seed=202 BA=0.739 FP=20 FN=12


MiniROCKET complete: voisard_2025 seed=314 BA=0.741 FP=24 FN=9


MiniROCKET complete: voisard_2025 seed=515 BA=0.728 FP=23 FN=11
{
  "accepted": false,
  "noninferior": false,
  "fp_and_fn_nonincreasing": false,
  "material_total_error_reduction": false,
  "mean_total_errors_baseline": 21.733333333333334,
  "mean_total_errors_candidate": 25.866666666666667,
  "relative_total_error_reduction": -0.19018404907975456,
  "worst_source_deltas": {
    "balanced_accuracy_delta": -0.01584587323301412,
    "specificity_delta": -0.04833333333333345,
    "sensitivity_delta": -0.0728682170542636,
    "auroc_delta": -0.001367989056087482,
    "mean_brier_delta": 0.01640899126431261
  },
  "paired_bootstrap": {
    "balanced_accuracy": {
      "mean_delta": -0.024516651621889075,
      "ci_low": -0.03972578676173314,
      "ci_high": -0.008931989462451644
    },
    "specificity": {
      "mean_delta": -0.024607843137254917,
      "ci_low": -0.05723393246187366,
      "ci_high": 0.008191721132897596
    },
    "sensitivity": {
      "mean_delta": -0.02442546010652

In [4]:
summary = rescue['metrics'].groupby(['method','held_out_source'], as_index=False).agg(
    auroc=('auroc','mean'), brier=('brier','mean'), balanced_accuracy=('balanced_accuracy','mean'),
    specificity=('specificity','mean'), sensitivity=('sensitivity','mean'),
    FP=('false_positives','mean'), FN=('false_negatives','mean'))
display(summary.round(4))
print(json.dumps(rescue['decision'], indent=2))


,method,held_out_source,auroc,brier,balanced_accuracy,specificity,sensitivity,FP,FN
0,deep_ensemble,felius_2024,0.8588,0.1551,0.7637,0.7647,0.7628,8.0,30.6
1,deep_ensemble,sint_maartenskliniek,0.8950,0.1387,0.8400,0.8000,0.8800,4.0,1.2
2,deep_ensemble,voisard_2025,0.9108,0.1338,0.8384,0.7583,0.9184,17.4,4.0
3,heterogeneous_equal,felius_2024,0.8575,0.1853,0.7479,0.8059,0.6899,6.6,40.0
4,heterogeneous_equal,sint_maartenskliniek,0.8980,0.1491,0.8050,0.7100,0.9000,5.8,1.0
5,heterogeneous_equal,voisard_2025,0.8950,0.1423,0.8156,0.7333,0.8980,19.2,5.0
6,minirocket_logistic,felius_2024,0.8597,0.2346,0.7257,0.8235,0.6279,6.0,48.0
7,minirocket_logistic,sint_maartenskliniek,0.8780,0.1697,0.7750,0.6500,0.9000,7.0,1.0
8,minirocket_logistic,voisard_2025,0.8299,0.1777,0.7433,0.7111,0.7755,20.8,11.0


{
  "accepted": false,
  "noninferior": false,
  "fp_and_fn_nonincreasing": false,
  "material_total_error_reduction": false,
  "mean_total_errors_baseline": 21.733333333333334,
  "mean_total_errors_candidate": 25.866666666666667,
  "relative_total_error_reduction": -0.19018404907975456,
  "worst_source_deltas": {
    "balanced_accuracy_delta": -0.01584587323301412,
    "specificity_delta": -0.04833333333333345,
    "sensitivity_delta": -0.0728682170542636,
    "auroc_delta": -0.001367989056087482,
    "mean_brier_delta": 0.01640899126431261
  },
  "paired_bootstrap": {
    "balanced_accuracy": {
      "mean_delta": -0.024516651621889075,
      "ci_low": -0.03972578676173314,
      "ci_high": -0.008931989462451644
    },
    "specificity": {
      "mean_delta": -0.024607843137254917,
      "ci_low": -0.05723393246187366,
      "ci_high": 0.008191721132897596
    },
    "sensitivity": {
      "mean_delta": -0.02442546010652323,
      "ci_low": -0.05718689025997995,
      "ci_high": 0.00

## Interpretation rule

The heterogeneous fusion failed: mean total errors increased from 21.73 to 25.87 per source/seed, mean FP increased by 0.73, and mean FN increased by 3.40. Balanced accuracy and calibration also worsened. Retain the deep ensemble and move to participant-level multiple-instance learning, which trains on bags of windows rather than assigning the participant diagnosis to every individual window. A binary output can be preserved, but genuinely near-zero FP and FN cannot be claimed by repeatedly optimizing against the same development participants.
